# 🛟 Document Scanner — Colab Fallback Runner

ใช้ตอน **Cloud Run ล่ม/ปิด** — รันไปป์ไลน์เดิม (Google Vision + Gemini → Typhoon) บน Colab แทน
โดย **import `main.py` ตัวเดียวกับ Cloud Run** จึงได้ตรรกะตรงกันเป๊ะ (โฟลเดอร์รายเดือน, แท็บ Error/บิลดี, ตัวนับโควตา ฯลฯ)

**ทำงานยังไง:** โพลล์โฟลเดอร์ Inbox เป็นระยะ → ประมวลผลทุกไฟล์ → เขียน Google Sheet → ย้ายเข้า `Success/YYYY-MM/` หรือ `Error/YYYY-MM/` → แจ้งผลทาง Telegram (เหมือน Cloud Run ทุกอย่าง)

> ⚠️ **อย่ารันพร้อม Cloud Run** — ถ้าทั้งคู่ทำงานพร้อมกันจะแย่งไฟล์กัน (double-process) ใช้ Colab นี้เฉพาะตอน Cloud Run ดับ
> ⚠️ ต้องมี **Service Account JSON key** (ตัวเดียวกับที่แชร์ Drive/Sheet/Vision ให้ Cloud Run) — เก็บใน **Colab Secrets** หรือ **Google Drive** ครั้งเดียว ไม่ต้องอัปโหลดทุกครั้ง
> ▶️ รันเซลล์ตามลำดับ 1 → 7

In [ ]:
# 1) ติดตั้งไลบรารี (ชุดเดียวกับ requirements.txt ของ Cloud Run)
!pip -q install "functions-framework==3.*" "requests>=2.31.0" "pillow>=10.0.0" "google-api-python-client>=2.100.0" "google-auth>=2.23.0" "google-cloud-vision>=3.14.0" "google-genai>=2.8.0" "openai==1.59.6" "pydantic>=2.0.0"
print("✅ ติดตั้งไลบรารีเสร็จ")

In [ ]:
# 2) โหลด Service Account key โดยไม่ต้อง "อัปโหลดทุกครั้ง" — เลือกวิธี A (แนะนำ) หรือ B
import os

# path ของ key ถ้าใช้วิธี B (เก็บไฟล์ไว้ใน Google Drive) — แก้ให้ตรงตำแหน่งจริง
DRIVE_KEY_PATH = "/content/drive/MyDrive/keys/scanner-service.json"  #@param {type:"string"}

try:
    # วิธี A: Colab Secrets — กดไอคอน 🔑 แถบซ้าย เพิ่ม secret ชื่อ SA_JSON วางเนื้อ JSON ทั้งก้อน
    #         แล้วเปิดสิทธิ์ Notebook access ให้ notebook นี้ (ตั้งครั้งเดียว ผูกกับบัญชี ใช้ได้ทุก notebook)
    from google.colab import userdata
    open("/content/sa.json", "w").write(userdata.get("SA_JSON"))
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/sa.json"
    print("✅ โหลด SA key จาก Colab Secrets (SA_JSON)")
except Exception as e:
    # วิธี B (สำรอง): อัปโหลด key ไว้ใน Google Drive ครั้งเดียว แล้ว mount มาใช้ (ตั้ง path ที่ DRIVE_KEY_PATH)
    print("ℹ️ ไม่พบ secret SA_JSON (", e, ") -> ใช้ไฟล์จาก Google Drive แทน")
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = DRIVE_KEY_PATH
    print("✅ ใช้ SA key จาก Drive:", DRIVE_KEY_PATH)

In [ ]:
# 3) กรอกค่าให้ตรงกับ env.yaml (ความลับ — กรอกในเซลล์นี้ ไม่ถูกบันทึกลงไฟล์ .ipynb)
TYPHOON_API_KEYS = ""  #@param {type:"string"}
GEMINI_API_KEY   = ""  #@param {type:"string"}
TELEGRAM_TOKEN   = ""  #@param {type:"string"}
TELEGRAM_CHAT_ID = ""  #@param {type:"string"}
SPREADSHEET_ID   = ""  #@param {type:"string"}
SHEET_TAB_NAME   = "Raw_Data"  #@param {type:"string"}
FOLDER_INBOX      = ""  #@param {type:"string"}
FOLDER_PROCESSING = ""  #@param {type:"string"}
FOLDER_SUCCESS    = ""  #@param {type:"string"}
FOLDER_ERROR      = ""  #@param {type:"string"}

import os
for _k in ["TYPHOON_API_KEYS","GEMINI_API_KEY","TELEGRAM_TOKEN","TELEGRAM_CHAT_ID",
           "SPREADSHEET_ID","SHEET_TAB_NAME","FOLDER_INBOX","FOLDER_PROCESSING",
           "FOLDER_SUCCESS","FOLDER_ERROR"]:
    os.environ[_k] = str(globals()[_k]).strip()
assert os.environ["SPREADSHEET_ID"] and os.environ["FOLDER_INBOX"], "ยังกรอก SPREADSHEET_ID / FOLDER_INBOX ไม่ครบ"
print("✅ ตั้งค่า env ครบ (โควตา/โมเดล Gemini ใช้ค่า default ใน main.py — ปรับเพิ่มเองได้ผ่าน os.environ)")

In [ ]:
# 4) เอา main.py ตัวจริงมา (วิธี A: อัปโหลด — ใช้ได้เสมอ แม้ repo เป็น private)
#    ลากไฟล์ main.py จากเครื่องคุณมาวางตอนขึ้นกล่องเลือกไฟล์
#    --- วิธี B (ทางเลือก) ถ้า repo เป็น public ดึงจาก GitHub แทน ---
#    !git clone --depth 1 https://github.com/Journey479/typhoon-bill-scanner.git /content/repo
#    import sys; sys.path.insert(0, "/content/repo")
from google.colab import files
_up = files.upload()
print("✅ ได้ไฟล์:", list(_up.keys()))

In [ ]:
# 5) โหลด main.py แล้วทดสอบเชื่อม Google
#    (ต้องรันเซลล์ 2-4 ก่อนเสมอ เพราะ main อ่าน env + สร้าง client ตอน import)
import importlib, main
importlib.reload(main)   # เผื่อแก้ env หรืออัปโหลด main ใหม่แล้วรันซ้ำ
drive, sheet, vision = main._services()
print("✅ เชื่อมต่อ Drive/Sheet สำเร็จ | เครื่องยนต์ Google:",
      "เปิด (Vision+Gemini)" if (main.GOOGLE_ENABLED and vision) else "ปิด -> ใช้ Typhoon ล้วน")

In [ ]:
# 6) ▶️ รันครั้งเดียว (กวาด Inbox ทั้งหมด 1 รอบ) — บังคับทำแม้ autorun=off
drive, sheet, vision = main._services()
s, e, t, total = main.process_cycle(drive, sheet, vision, manual=True)
print(f"เสร็จ 1 รอบ -> สำเร็จ {s} | error {e} | timeout {t} | ทั้งหมด {total}")

In [ ]:
# 7) 🔁 โหมด fallback — โพลล์ Inbox อัตโนมัติ (หยุดด้วยปุ่ม ⏹ ของ Colab)
#    เคารพสวิตช์ autorun เดียวกับ Telegram (สั่ง autorun:off = พักชั่วคราว)
import time
POLL_SECONDS = 60  #@param {type:"integer"}
print(f"เริ่มโหมด fallback — โพลล์ทุก {POLL_SECONDS}s | กด ⏹ เพื่อหยุด")
while True:
    try:
        drive, sheet, vision = main._services()
        if main.read_autorun_state(drive) == "on":
            s, e, t, total = main.process_cycle(drive, sheet, vision, manual=False)
            if total:
                print(time.strftime("%H:%M:%S"), f"-> สำเร็จ {s} | error {e} | timeout {t} | รวม {total}")
        else:
            print(time.strftime("%H:%M:%S"), "autorun=off — พัก")
    except Exception as ex:
        print(time.strftime("%H:%M:%S"), "⚠️ รอบนี้ error:", ex)
    time.sleep(POLL_SECONDS)